<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: Many metrics and many tests

**[Many metrics and many tests](https://learning.nextia-ai.com/courses/experiments/m04/many-metrics-and-many-tests/)** · Experiments and Causal Impact: Did Our Change Work? · Module 4, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure, by simulation, how often a test with 20 metrics finds a false winner, and to correct for it with Bonferroni and Benjamini–Hochberg.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | p-values, from [The result and its uncertainty](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/); peeking, from [Peeking](https://learning.nextia-ai.com/courses/experiments/m04/peeking/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m04/many-metrics-and-many-tests/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M04-L02-many-metrics-and-many-tests/many-metrics-and-many-tests-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: the project folder

This notebook makes its own data with small simulations, so it downloads nothing. Run the next cell. It makes a folder `larkfield-experiments` and works in it.

You should see `Ready: 0 files in data/.`

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One A/A test with 20 metrics

An A/A test shows both groups the same page, so no metric has a real effect. The next cell simulates one A/A test with 2,000 customers per group and 20 metrics, and computes Welch's t-test for each metric.

> **Predict.** How many of the 20 metrics will show p < 0.05?

In [ ]:
def aa_metrics(n_tests, n=2000, metrics=20, shared=0.0, seed=0):
    """p-values of n_tests A/A tests, each with `metrics` metrics. `shared` links the metrics:
    0 = independent; 0.6 = one hidden "how engaged" value moves every metric a little."""
    rng = np.random.default_rng(seed)
    p = np.empty((n_tests, metrics))
    for t in range(n_tests):
        ea, eb = rng.normal(0, 1, n), rng.normal(0, 1, n)
        a = shared * ea[:, None] + np.sqrt(1 - shared**2) * rng.normal(0, 1, (n, metrics))
        b = shared * eb[:, None] + np.sqrt(1 - shared**2) * rng.normal(0, 1, (n, metrics))
        p[t] = stats.ttest_ind(b, a, equal_var=False).pvalue
    return p

p_one = aa_metrics(1)[0]
print(np.round(np.sort(p_one), 4))
print("metrics with p < 0.05:", int((p_one < 0.05).sum()))

> **Check.** You should see 20 sorted p-values and `metrics with p < 0.05: 2`.

Nothing changed, and still 2 metrics "won". With 20 chances at 5% each, about one false winner is normal.

## 2. Many A/A tests: how often is there at least one winner?

In [ ]:
p_ind = aa_metrics(1000)
p_cor = aa_metrics(1000, shared=0.6, seed=1)
print(f"independent metrics: {(p_ind < 0.05).any(axis=1).mean():.1%} of tests have at least one winner")
print(f"linked metrics:      {(p_cor < 0.05).any(axis=1).mean():.1%} of tests have at least one winner")
print(f"the formula 1 - 0.95**20 = {1 - 0.95**20:.1%}")

> **Check.** You should see `independent metrics: 64.9%`, `linked metrics: 50.9%` and `the formula 1 - 0.95**20 = 64.2%`.

Real metrics move together (a customer who views more pages also adds more to the cart), so they give fewer independent chances. The rate is still far above 5%. The course's 300 runs of the many-metrics scenario give 56%.

## 3. Two corrections

**Bonferroni** divides 0.05 by the number of metrics: with 20 metrics, a winner needs p < 0.0025. **Benjamini–Hochberg** (BH) sorts the p-values and compares the k-th smallest with k / 20 × 0.05. It keeps every metric up to the largest k that passes.

In [ ]:
def bonferroni(p, alpha=0.05):
    p = np.asarray(p)
    return p < alpha / len(p)

def benjamini_hochberg(p, alpha=0.05):
    p = np.asarray(p)
    m = len(p)
    order = np.argsort(p)
    below = p[order] <= alpha * np.arange(1, m + 1) / m
    keep = np.zeros(m, bool)
    if below.any():
        keep[order[: np.nonzero(below)[0].max() + 1]] = True
    return keep

for name, p in [("independent", p_ind), ("linked", p_cor)]:
    bon = np.mean([bonferroni(r).any() for r in p])
    bh = np.mean([benjamini_hochberg(r).any() for r in p])
    print(f"{name}: Bonferroni {bon:.1%}, Benjamini-Hochberg {bh:.1%} of tests with a false winner")

> **Check.** You should see `independent: Bonferroni 5.9%, Benjamini-Hochberg 6.2%` and `linked: Bonferroni 3.5%, Benjamini-Hochberg 3.5%`.

Both corrections bring the false winners back to about 5% of tests, or fewer. In an A/A test every winner is false, so here they behave alike. With real effects, BH keeps more of them.

## 4. Your turn: Benjamini–Hochberg by hand

> **Your turn.** Five metrics of one test have the p-values below (made up for the exercise). Put in `bh_limits` the five BH limits k / 5 × 0.05, and in `kept` the number of metrics that BH keeps. Then compare with Bonferroni, which keeps only p < 0.01.

In [ ]:
p5 = [0.004, 0.012, 0.030, 0.30, 0.70]   # already sorted
bh_limits = [round(k / 5 * 0.05, 3) for k in range(1, 6)]
passes = [p <= limit for p, limit in zip(p5, bh_limits)]
kept = max(k + 1 for k, ok in enumerate(passes) if ok)
print(bh_limits, passes, kept)

In [ ]:
expect_hash('bh_limits', bh_limits, '7fb23deb1e98410c')

In [ ]:
expect('kept', kept, 3)

BH keeps three metrics; Bonferroni keeps one. BH controls the share of false winners among the winners, not the chance of any false winner, so it is less strict.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m04/many-metrics-and-many-tests/) explains when to use each correction and why the primary metric is chosen before the test, and has the knowledge checks that count toward your certificate. In the project, `xp metrics many-metrics` runs both corrections on the 20 metrics of a planted test.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Novelty and learning effects](https://learning.nextia-ai.com/courses/experiments/m04/novelty-and-learning-effects/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m04/many-metrics-and-many-tests/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.